# Множества. Отображения и функции

### *Лекция читает: профессор Звонарёв И.Б., кафедра нейросетей и сравнительно честной математики*

> Дорогой студент! Если ты открыл этот ноутбук и ничего не понял — поздравляю, ты нормальный человек. Теория множеств придумана Георгом Кантором в XIX веке, и его коллеги тогда тоже ничего не поняли — некоторые даже обиделись и пытались уволить Кантора из университета. Так что ты в хорошей компании.
>
> Сейчас я тебе всё разжую. По ходу дела мы:
> 1. Поймём, что такое множество (без воды, на картинках).
> 2. Научимся их складывать, вычитать и пересекать — как в детстве с кубиками.
> 3. Разберёмся с функциями и их типами (инъекция, сюръекция, биекция — три страшных слова, за которыми стоят простые картинки).
> 4. Дойдём до **диагонального аргумента Кантора** — это когда один человек за 5 минут доказал, что бесконечности бывают разные, и некоторые «больше» других.
> 5. В конце — **тест**. Пройдёшь — гуляй с миром. Не пройдёшь — возвращайся к началу.

**Уровень:** с нуля. Если знаешь, что $2+2=4$, ты готов.

**Время:** ~2 часа. Можно с чаем.

---

## План лекции

| # | Тема | Зачем это в ML |
|---|---|---|
| 1 | Что такое множество | Датасет = множество объектов |
| 2 | Обозначения и символы | Читать чужие статьи без словаря |
| 3 | Операции над множествами | Дедупликация, фильтрация, join |
| 4 | Подмножества и булеан | Классы, подклассы, grid search |
| 5 | Отображения и функции | Любая нейросеть — это функция |
| 6 | Типы: инъекция, сюръекция, биекция | Loss-функции, эмбеддинги, автокодеры |
| 7 | Мощность множества | Размерность пространства признаков |
| 8 | Диагональный аргумент Кантора | Почему $\mathbb{R}$ «больше» $\mathbb{N}$ |
| 9 | Задачи с лекции ВК | $\|\mathbb{R}^k\|$, $\|\mathbb{Q}^k\|$, последовательности |
| 10 | Связь с нейросетями | Что из этого реально нужно в ML |
| 11 | Проверочный тест | Закрепить и проверить себя |

---

## Подготовка окружения

In [ ]:
# ============================================================
# Подготовка окружения — всё, что нам понадобится
# ============================================================
# Если чего-то нет — pip install <name>

import math                          # Базовая математика: sqrt, pi, log
import random                        # Случайные числа (для демо)
import itertools                     # combinations, permutations — для булеана
from fractions import Fraction       # Точные рациональные числа (без потерь float)

import numpy as np                   # Векторные операции
import matplotlib.pyplot as plt      # Графики и схемы
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle, FancyBboxPatch
from matplotlib_venn import venn2, venn3   # Диаграммы Венна
import matplotlib.font_manager as fm

# Настройка шрифтов для кириллицы в подписях
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

# Воспроизводимость (одинаковые картинки при каждом запуске)
random.seed(42)
np.random.seed(42)

print("Профессор готов к лекции. Садитесь поудобнее.")

---

## 1. Что такое множество?

> **Множество** — это любая совокупность определённых и различимых объектов, мыслимая как единое целое. *(Г. Кантор, 1895)*

Профессорский пересказ: **множество — это коробка с разными вещами.** Вещи — «элементы». Коробка — «множество».

### Три ключевых правила

1. **В коробке нет дубликатов.** Если положишь два одинаковых носка — получишь один носок в коробке. Математически: $\{1, 1, 2\} = \{1, 2\}$.
2. **Порядок не важен.** $\{\text{чай}, \text{сахар}\} = \{\text{сахар}, \text{чай}\}$. Это не список, это мешок.
3. **Чёткое членство.** Либо предмет в коробке, либо нет. Никаких «наполовину». (Нечёткие множества Заде — отдельная история для продвинутых.)

### Примеры из жизни ML-щика

| Множество | Что внутри | Мощность (размер) |
|---|---|---|
| Множество токенов в словаре GPT-4 | слова и подслова | ~100 000 |
| Множество пикселей в картинке 224×224 | пары (x, y) | 50 176 |
| Множество классов ImageNet | названия объектов | 1000 |
| Множество весов ResNet-50 | вещественные числа | ~25 000 000 |
| Множество обучающих примеров | картинки с подписями | ~1 000 000 |

> **Зачем это в ML.** Каждый датасет — это множество объектов. Каждая нейросеть — это функция, отображающая множество объектов в множество ответов. Если ты не понимаешь множества — ты не понимаешь, над чем твоя сеть работает.

In [ ]:
# ============================================================
# Визуализация: множество как коробка с шариками
# ============================================================
fig, ax = plt.subplots(figsize=(9, 4))

# Рисуем "коробку" — большой прямоугольник
box = FancyBboxPatch((0.5, 0.5), 8, 3, boxstyle="round,pad=0.1",
                      linewidth=2, edgecolor='#2c3e50', facecolor='#fef9e7')
ax.add_patch(box)
ax.text(4.5, 3.85, 'A = {1, 2, 3, 5, 8}', fontsize=14, ha='center',
        fontweight='bold', color='#2c3e50')

# Шарики внутри — элементы множества
elements = [(1.5, 1.8, '1'), (3.0, 1.8, '2'), (4.5, 1.8, '3'),
            (6.0, 1.8, '5'), (7.5, 1.8, '8')]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
for (x, y, lbl), c in zip(elements, colors):
    circle = Circle((x, y), 0.4, color=c, alpha=0.85, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, lbl, fontsize=14, ha='center', va='center',
            color='white', fontweight='bold', zorder=4)

# Подписи
ax.text(4.5, 0.2, 'Пять элементов. Каждый — разный. Порядок не важен.',
        ha='center', fontsize=10, style='italic', color='#7f8c8d')

ax.set_xlim(0, 9)
ax.set_ylim(0, 4.3)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Множество A = {1, 2, 3, 5, 8} — коробка с пятью шариками',
             fontsize=12, pad=10)
plt.tight_layout()
plt.show()

---

## 2. Обозначения и символы — расшифровка каждого

Это самый важный раздел для понимания. Я **каждый символ** разберу.

### 2.1. Принадлежность: $x \in A$

Читается: **«икс принадлежит а»** или **«икс лежит в а»**.

| Символ | Откуда взялся | Что значит |
|---|---|---|
| $x$ | латинская буква | любой объект (число, строка, точка) |
| $\in$ | от греч. epsilon $\epsilon$ | «является элементом» |
| $A$ | заглавная буква | имя множества |

**Пример:** $2 \in \{1, 2, 3\}$ — двойка лежит в множестве $\{1, 2, 3\}$. Это **истина** (True).

### 2.2. Непринадлежность: $x \notin A$

Читается: **«икс не принадлежит а»**.

Символ $\notin$ — это $\in$ с перечёркнутой чертой. Логично.

**Пример:** $4 \notin \{1, 2, 3\}$ — четвёрки нет в множестве $\{1, 2, 3\}$. Тоже **истина**.

### 2.3. Мощность: $|A|$

Читается: **«мощность а»** или **«размер а»**.

Вертикальные черты $|\ldots|$ — это как абсолютная величина $|x|$ для чисел, только для множеств. Если $A$ — коробка, то $|A|$ — сколько в ней шариков.

**Пример:** $|\{1, 2, 3\}| = 3$ — в множестве три элемента.

> **Тонкость.** Для бесконечных множеств $|A|$ — это уже не число, а **кардинальное число** (трансконечное). $|\mathbb{N}| = \aleph_0$ (читается «алеф-ноль»). Но об этом ниже.

### 2.4. Пустое множество: $\emptyset$

Читается: **«пустое множество»** или **«эмпти сет»**.

Символ $\emptyset$ — это скандинавская буква, придуманная группой Бурбаки в 1939 году. Это **множество без элементов**, а не «ничего». Аналогия: пустая коробка — это всё ещё коробка.

**Свойство:** $|\emptyset| = 0$.

**Подвох:** $\emptyset \neq \{\emptyset\}$. Первое — пустое множество (ноль элементов). Второе — множество, содержащее один элемент (этот элемент — пустое множество). Это как разница между «пустой коробкой» и «коробкой, в которой лежит другая пустая коробка».

### 2.5. Set-builder notation: $\{x : P(x)\}$

Читается: **«множество всех икс, для которых пэ от икс истинно»**.

| Часть | Что значит |
|---|---|
| $\{\ldots\}$ | фигурные скобки = «это множество» |
| $x$ | переменная — пробегает все возможные значения |
| $:$ или $|$ | «такой что» (separator) |
| $P(x)$ | условие — функция, возвращающая True/False |

**Пример:** $\{n \in \mathbb{N} : n < 4\} = \{0, 1, 2, 3\}$.

Читаем по частям: «возьмём все $n$ из натуральных, такие что $n < 4$». Получаем $\{0, 1, 2, 3\}$ (ноль включён по соглашению курса).

### 2.6. Стандартные числовые множества

| Символ | Название | Что внутри | Примеры |
|---|---|---|---|
| $\mathbb{N}$ | Натуральные | $0, 1, 2, 3, \ldots$ | $0, 5, 42$ |
| $\mathbb{Z}$ | Целые | $\ldots, -2, -1, 0, 1, 2, \ldots$ | $-7, 0, 100$ |
| $\mathbb{Q}$ | Рациональные | дроби $p/q$ | $1/2, -3/7, 5$ |
| $\mathbb{R}$ | Действительные | все точки числовой прямой | $\sqrt{2}, \pi, -1.41$ |
| $\mathbb{C}$ | Комплексные | $a + bi$ | $1+2i, 0, \pi - i$ |

> **Почему буквы «жирные».** $\mathbb{N}$, $\mathbb{R}$ и т.д. — это так называемые **blackboard bold** («двойные chalkboard letters»). Раньше математики на доске писали обычную $N$ и для отличия от переменной просто перечёркивали — получалась «двойная» буква. В печатном тексте это стало особым шрифтом.
>
> **Соглашение ВК-курса:** $0 \in \mathbb{N}$. В английской традиции часто $\mathbb{N}$ начинается с 1. В русском курсе — с 0. Это ВАЖНО для расчётов.

---

## 3. Операции над множествами

Это как арифметика, только для мешков с шариками.

### 3.1. Четыре основные операции

| Операция | Обозн. | Python | Что делаем |
|---|---|---|---|
| Объединение | $A \cup B$ | `A \| B` | всё из A **и** всё из B |
| Пересечение | $A \cap B$ | `A & B` | только то, что есть **и там, и там** |
| Разность | $A \setminus B$ | `A - B` | то, что в A, **но не** в B |
| Симм. разность | $A \triangle B$ | `A ^ B` | то, что **только в одном** из них |

### 3.2. Формулы — разбираем по символам

**Объединение:** $A \cup B = \{x : x \in A \lor x \in B\}$

| Символ | Что значит |
|---|---|
| $A \cup B$ | «объединение A и B», символ $\cup$ похож на cup (чашка) — собираем всё |
| $\{x : \ldots\}$ | множество всех $x$, для которых выполняется условие справа |
| $x \in A$ | $x$ лежит в A |
| $\lor$ | логическое «ИЛИ» (от лат. vel) |
| $x \in B$ | $x$ лежит в B |

Читаем: «объединение A и B — это множество всех $x$, которые лежат в A **ИЛИ** в B».

**Пересечение:** $A \cap B = \{x : x \in A \land x \in B\}$

Символ $\land$ — логическое «И» (от англ. AND). $\cap$ похож на перевёрнутую чашку — «пересечение».

**Разность:** $A \setminus B = \{x : x \in A \land x \notin B\}$

Символ $\setminus$ — обратный слэш. Понятно: «убери из A то, что есть в B».

**Симметрическая разность:** $A \triangle B = (A \setminus B) \cup (B \setminus A)$

Символ $\triangle$ — треугольник (delta). Это «исключающее ИЛИ» для множеств: элементы, которые лежат **ровно в одном** из двух.

### 3.3. Законы де Моргана

Эти законы — твой друг, когда надо упрощать выражения:

$$\overline{A \cup B} = \overline{A} \cap \overline{B}$$
$$\overline{A \cap B} = \overline{A} \cup \overline{B}$$

Читаем первый: **дополнение объединения = пересечение дополнений**. Звучит странно, но картинка всё проясняет.

> **Зачем это в ML.** Допустим, ты фильтруешь датасет: «оставить картинки, которые НЕ (кошки ИЛИ собаки)». По де Моргану это = «НЕ кошки И НЕ собаки». Часто проще реализовать.

In [ ]:
# ============================================================
# Визуализация: четыре операции над множествами
# ============================================================
A = {1, 2, 3, 4, 5}
B = {4, 5, 6, 7, 8}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
ops = [
    ('A ∪ B  (объединение)', A | B, '#2ecc71'),
    ('A ∩ B  (пересечение)', A & B, '#e74c3c'),
    ('A \\ B  (разность)', A - B, '#3498db'),
    ('A △ B  (симм. разность)', A ^ B, '#f39c12'),
]

for ax, (title, result, color) in zip(axes.flat, ops):
    v = venn2(subsets=(len(A - B), len(B - A), len(A & B)),
              set_labels=('A', 'B'), ax=ax)
    # Подсветим нужные области
    if '∪' in title:
        for pid in ['10', '01', '11']:
            if v.get_patch_by_id(pid):
                v.get_patch_by_id(pid).set_color(color)
                v.get_patch_by_id(pid).set_alpha(0.6)
    elif '∩' in title:
        for pid in ['10', '01']:
            if v.get_patch_by_id(pid):
                v.get_patch_by_id(pid).set_color('lightgray')
                v.get_patch_by_id(pid).set_alpha(0.3)
        if v.get_patch_by_id('11'):
            v.get_patch_by_id('11').set_color(color)
            v.get_patch_by_id('11').set_alpha(0.7)
    elif 'разность' in title:
        for pid in ['01', '11']:
            if v.get_patch_by_id(pid):
                v.get_patch_by_id(pid).set_color('lightgray')
                v.get_patch_by_id(pid).set_alpha(0.3)
        if v.get_patch_by_id('10'):
            v.get_patch_by_id('10').set_color(color)
            v.get_patch_by_id('10').set_alpha(0.7)
    elif 'симм' in title:
        for pid in ['10', '01']:
            if v.get_patch_by_id(pid):
                v.get_patch_by_id(pid).set_color(color)
                v.get_patch_by_id(pid).set_alpha(0.6)
        if v.get_patch_by_id('11'):
            v.get_patch_by_id('11').set_color('lightgray')
            v.get_patch_by_id('11').set_alpha(0.3)
    ax.set_title(f'{title}\n= {sorted(result)}', fontsize=12)

plt.suptitle('Четыре операции над множествами A={1,2,3,4,5} и B={4,5,6,7,8}',
             fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

# Дополнительно: проверка законов де Моргана
U = set(range(1, 11))  # универсум {1..10}
print('Универсум U =', sorted(U))
print()
print('Проверка первого закона де Моргана: ¬(A ∪ B) = ¬A ∩ ¬B')
left = U - (A | B)                    # ¬(A ∪ B)
right = (U - A) & (U - B)             # ¬A ∩ ¬B
print(f'  ¬(A ∪ B) = {sorted(left)}')
print(f'  ¬A ∩ ¬B = {sorted(right)}')
print(f'  Совпадает? {left == right}')

print()
print('Проверка второго закона де Моргана: ¬(A ∩ B) = ¬A ∪ ¬B')
left = U - (A & B)
right = (U - A) | (U - B)
print(f'  ¬(A ∩ B) = {sorted(left)}')
print(f'  ¬A ∪ ¬B = {sorted(right)}')
print(f'  Совпадает? {left == right}')

---

## 4. Подмножества и булеан

### 4.1. Подмножество: $A \subseteq B$

Читается: **«а — подмножество бэ»** или **«а содержится в бэ»**.

| Символ | Что значит |
|---|---|
| $\subseteq$ | подмножество, возможно равное (≤ для множеств) |
| $\subset$ | **собственное** подмножество, строго меньшее (< для множеств) |

**Определение:** $A \subseteq B$ означает, что **каждый** элемент $A$ лежит в $B$:

$$A \subseteq B \iff \forall x \;(x \in A \Rightarrow x \in B)$$

Разберём по символам:
- $\forall x$ — «для любого $x$» (от англ. **for all**)
- $\Rightarrow$ — «следовательно» (импликация)
- $x \in A \Rightarrow x \in B$ — «если $x$ лежит в A, то $x$ лежит и в B»

### 4.2. Равенство множеств

$$A = B \iff (A \subseteq B) \land (B \subseteq A)$$

Чтобы доказать $A = B$, надо доказать **два** включения: $A \subseteq B$ и $B \subseteq A$. Это стандартный приём в математических доказательствах.

### 4.3. Булеан: $\mathcal{P}(A)$

**Булеан** множества $A$ — это множество **всех** его подмножеств (включая $\emptyset$ и само $A$).

$$\mathcal{P}(A) = \{B : B \subseteq A\}$$

**Теорема.** Если $|A| = n$, то $|\mathcal{P}(A)| = 2^n$.

**Почему $2^n$?** Для каждого элемента $A$ есть два варианта: он **в подмножестве** или **не в подмножестве**. Умножаем $n$ двоек: $2^n$.

> **Зачем это в ML.** Булеан — это **все возможные фичи**, которые можно построить из $n$ бинарных признаков. Если у тебя 10 бинарных признаков, то $|\mathcal{P}| = 2^{10} = 1024$ возможных комбинации. Это и есть пространство, в котором работает логистическая регрессия.

In [ ]:
# ============================================================
# Визуализация: булеан множества {1, 2, 3} как дерево
# ============================================================
import itertools

A = [1, 2, 3]
subsets_by_size = {}
for r in range(len(A) + 1):
    subsets_by_size[r] = list(itertools.combinations(A, r))

fig, ax = plt.subplots(figsize=(12, 6))

# Координаты: уровень = размер подмножества
positions = {}  # subset -> (x, y)
for size, subs in subsets_by_size.items():
    n = len(subs)
    for i, s in enumerate(subs):
        x = i + 0.5 * (4 - n)  # центрируем
        y = -size  # ниже = больше элементов
        positions[s] = (x, y)

# Рисуем рёбра (между подмножествами, отличающимися одним элементом)
for s1, (x1, y1) in positions.items():
    for s2, (x2, y2) in positions.items():
        if y2 - y1 == 1 and set(s2) > set(s1) and len(set(s2) - set(s1)) == 1:
            ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.4, zorder=1)

# Рисуем узлы
for s, (x, y) in positions.items():
    label = '{' + ', '.join(map(str, s)) + '}' if s else '∅'
    color = '#3498db' if s else '#95a5a6'
    size = 1400 if len(s) <= 1 else 1800
    ax.scatter(x, y, s=size, color=color, alpha=0.85, zorder=3, edgecolor='white', linewidth=2)
    ax.text(x, y, label, fontsize=11, ha='center', va='center',
            color='white', fontweight='bold', zorder=4)

# Подписи уровней
for size in range(len(A) + 1):
    ax.text(-0.8, -size, f'размер {size}\n({len(subsets_by_size[size])} шт.)',
            fontsize=10, ha='right', va='center', color='#7f8c8d')

ax.set_xlim(-2, 5)
ax.set_ylim(-3.5, 0.8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Булеан P({1, 2, 3}) — все 2³ = 8 подмножеств,\nупорядоченные по размеру (дерево Хассе)',
             fontsize=12, pad=15)
plt.tight_layout()
plt.show()

print(f'Проверка: |A| = {len(A)}, |P(A)| = 2^{len(A)} = {2**len(A)}')
print(f'Подмножества: {sorted([set(s) if s else set() for s in itertools.chain.from_iterable(subsets_by_size.values())], key=lambda x: (len(x), sorted(x)))}')

---

## 5. Отображения и функции

### 5.1. Определение

Пусть $A$ и $B$ — множества. **Отображение** $f$ из $A$ в $B$ (запись $f: A \to B$) — это правило, сопоставляющее **каждому** элементу $A$ **ровно один** элемент $B$.

$$f: A \to B \quad \text{означает} \quad \forall x \in A \;\exists!\, y \in B : y = f(x)$$

Разбираем по символам:

| Символ | Что значит |
|---|---|
| $f: A \to B$ | $f$ — отображение из $A$ в $B$; стрелка $\to$ = направление |
| $\forall x \in A$ | «для каждого $x$ из $A$» |
| $\exists!$ | «существует **ровно один**» (символ $\exists$ — existence, воскл. знак — единственность) |
| $y \in B$ | «$y$ лежит в $B$» |
| $y = f(x)$ | «$y$ равно эф от икс» — значение функции в точке $x$ |

### 5.2. Словарик: функция / отображение / оператор / функционал

В школе это синонимы. В высшей математике — разные вещи:

| Термин | Что отображает | Пример |
|---|---|---|
| **Функция** | числа $\to$ числа | $f(x) = x^2$ |
| **Отображение** | любое множество $\to$ любое | хеширование строки в int |
| **Оператор** | функции $\to$ функции | $\frac{d}{dx}$ — производная |
| **Функционал** | функции $\to$ числа | $\int_0^1 f(x)\,dx$ |

### 5.3. Образ, прообраз, график

| Термин | Обозн. | Что это |
|---|---|---|
| **График** | $\Gamma_f$ | $\{(x, f(x)) : x \in A\}$ — пары «вход-выход» |
| **Образ множества** $X$ | $f(X)$ | $\{f(x) : x \in X\}$ — куда переходит $X$ |
| **Образ всего $A$** | $\mathrm{Im}(f)$ | $f(A)$ — множество значений функции |
| **Прообраз множества** $Y$ | $f^{-1}(Y)$ | $\{x \in A : f(x) \in Y\}$ — что переходит в $Y$ |

> ⚠️ **Важно:** $f^{-1}(Y)$ — это **не обратная функция**, а **прообраз**. Это разные вещи. Обратная функция существует только у биекций (см. след. раздел), а прообраз определён всегда.
>
> **Зачем это в ML.** Нейросеть — это отображение $f: \mathbb{R}^n \to \mathbb{R}^m$. Образ — её выходы. Функция потерь $L$ — функционал: отображает функцию (нейросеть) в число. Градиент — оператор: отображает функцию в другую функцию.

In [ ]:
# ============================================================
# Визуализация: отображение f : A -> B, стрелочки
# ============================================================
fig, ax = plt.subplots(figsize=(11, 5))

# Множество A (слева) и B (справа)
A_elements = [('a', 1, 3), ('b', 1, 2), ('c', 1, 1), ('d', 1, 0)]
B_elements = [('x', 5, 3.5), ('y', 5, 2.5), ('z', 5, 1.5), ('w', 5, 0.5)]

# Отображение: каждый элемент A -> элемент B
f_map = {'a': 'x', 'b': 'y', 'c': 'y', 'd': 'z'}  # не инъекция (b,c -> y), не сюръекция (нет w)

# Рисуем "облака" множеств
from matplotlib.patches import Ellipse
ell_A = Ellipse((1, 1.5), 1.0, 4.2, alpha=0.15, color='#3498db')
ell_B = Ellipse((5, 2.0), 1.0, 4.2, alpha=0.15, color='#e74c3c')
ax.add_patch(ell_A)
ax.add_patch(ell_B)

# Подписи множеств
ax.text(1, 4.1, 'A', fontsize=18, ha='center', fontweight='bold', color='#2c3e50')
ax.text(5, 4.5, 'B', fontsize=18, ha='center', fontweight='bold', color='#2c3e50')

# Элементы A
for name, x, y in A_elements:
    ax.scatter(x, y, s=800, color='#3498db', zorder=3, edgecolor='white', linewidth=2)
    ax.text(x, y, name, fontsize=14, ha='center', va='center', color='white',
            fontweight='bold', zorder=4)

# Элементы B
for name, x, y in B_elements:
    # Подсветим элементы B, в которые кто-то переходит
    color = '#e74c3c' if name in f_map.values() else '#bdc3c7'
    ax.scatter(x, y, s=800, color=color, zorder=3, edgecolor='white', linewidth=2)
    ax.text(x, y, name, fontsize=14, ha='center', va='center', color='white',
            fontweight='bold', zorder=4)

# Стрелки f
pos_A = {n: (x, y) for n, x, y in A_elements}
pos_B = {n: (x, y) for n, x, y in B_elements}

for src, dst in f_map.items():
    x1, y1 = pos_A[src]
    x2, y2 = pos_B[dst]
    arrow = FancyArrowPatch((x1 + 0.18, y1), (x2 - 0.18, y2),
                             arrowstyle='->', mutation_scale=20,
                             color='#2c3e50', linewidth=2, zorder=2)
    ax.add_patch(arrow)

# Подпись функции
ax.text(3, 4.5, 'f', fontsize=20, ha='center', fontweight='bold',
        color='#8e44ad', style='italic')
ax.text(3, 0.0, 'f: A → B', fontsize=14, ha='center', color='#2c3e50')

# Анализ
ax.text(3, -0.7, 'Замечания:\n'
                 '• b и c → y: f НЕ инъекция\n'
                 '• w не имеет прообраза: f НЕ сюръекция\n'
                 '• Значит, f НЕ биекция',
        fontsize=10, ha='center', va='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#fef9e7', edgecolor='#f39c12'))

ax.set_xlim(-0.5, 6.5)
ax.set_ylim(-1.5, 5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Отображение f: A → B — каждый элемент A переходит ровно в один элемент B',
             fontsize=12, pad=10)
plt.tight_layout()
plt.show()

---

## 6. Инъекция, сюръекция, биекция

Это **три типа** отображений. Запомнишь — будешь щёлкать задачи как орешки.

### 6.1. Инъекция («в жизнь — один»)

> **Инъекция** — это отображение, где **разным $x$ соответствуют разные $y$**.

$$f(x_1) = f(x_2) \Rightarrow x_1 = x_2$$

**Разбираем:**
- $f(x_1) = f(x_2)$ — два значения функции совпали
- $\Rightarrow$ — значит
- $x_1 = x_2$ — это был один и тот же $x$

**Иначе говоря:** нет двух разных $x$, которые дают одинаковый $f(x)$.

**Пример инъекции:** $f(x) = x + 1$. Разные $x$ дают разные $x+1$.

**Контрпример:** $f(x) = x^2$. Здесь $f(2) = f(-2) = 4$ — два разных $x$ дают один $y$. Не инъекция!

### 6.2. Сюръекция («на»)

> **Сюръекция** — это отображение, где **каждый $y$ из $B$ достижим**.

$$\forall y \in B \;\exists x \in A : f(x) = y$$

**Разбираем:**
- $\forall y \in B$ — для любого $y$ из области прибытия
- $\exists x \in A$ — существует $x$ в области определения
- $f(x) = y$ — такой что $f(x)$ равно этому $y$

**Иначе говоря:** в $B$ нет «лишних» элементов — все используются.

**Пример сюръекции:** $f: \mathbb{R} \to \mathbb{R}$, $f(x) = x^3$. Каждое $y$ достижимо (кубический корень существует из любого числа).

**Контрпример:** $f: \mathbb{R} \to \mathbb{R}$, $f(x) = x^2$. Отрицательные $y$ не достижимы. Не сюръекция.

### 6.3. Биекция («взаимно однозначное»)

> **Биекция** = инъекция + сюръекция.

| Тип | Условие | Аналогия |
|---|---|---|
| Инъекция | разные $x$ → разные $y$ | разные люди = разные номера телефонов |
| Сюръекция | все $y$ заняты | у каждого телефона есть владелец |
| Биекция | оба условия | у каждого человека ровно один телефон, и наоборот |

**Биекция — это «словарь»:** по $x$ можно найти $y$, по $y$ — $x$. Обратное отображение $f^{-1}$ существует.

> **Зачем это в ML.** Если ты строишь **эмбеддинги** для токенов, хочешь, чтобы разным токенам соответствовали разные векторы (инъекция). Если хочешь, чтобы по вектору можно было восстановить токен (decoder), нужна биекция. Автокодеры учат биекцию «сжать-восстановить».

In [ ]:
# ============================================================
# Визуализация: инъекция, сюръекция, биекция — три примера
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cases = [
    {
        'title': 'Инъекция (не сюръекция)',
        'A': ['a', 'b', 'c'],
        'B': ['x', 'y', 'z', 'w'],
        'f': {'a': 'x', 'b': 'y', 'c': 'z'},  # w не задействован
        'subtitle': 'Разные x → разные y\nНО w "лишний"'
    },
    {
        'title': 'Сюръекция (не инъекция)',
        'A': ['a', 'b', 'c', 'd'],
        'B': ['x', 'y', 'z'],
        'f': {'a': 'x', 'b': 'y', 'c': 'y', 'd': 'z'},  # b и c -> y
        'subtitle': 'Все y заняты\nНО b,c → один y'
    },
    {
        'title': 'БИЕКЦИЯ (и то, и другое)',
        'A': ['a', 'b', 'c'],
        'B': ['x', 'y', 'z'],
        'f': {'a': 'x', 'b': 'y', 'c': 'z'},
        'subtitle': 'Разные x → разные y\nИ все y заняты'
    },
]

for ax, case in zip(axes, cases):
    A_els = case['A']
    B_els = case['B']
    f_map = case['f']

    n_a = len(A_els)
    n_b = len(B_els)

    # Координаты элементов
    pos_A = {a: (1, n_a - i) for i, a in enumerate(A_els)}
    pos_B = {b: (4, n_b - i + (n_a - n_b) / 2) for i, b in enumerate(B_els)}

    # Облака
    ax.add_patch(Ellipse((1, (n_a + 1) / 2), 1.2, n_a + 1, alpha=0.15, color='#3498db'))
    ax.add_patch(Ellipse((4, (n_b + 1) / 2 + (n_a - n_b) / 2), 1.2, n_b + 1, alpha=0.15, color='#e74c3c'))

    # Элементы A
    for name, (x, y) in pos_A.items():
        ax.scatter(x, y, s=600, color='#3498db', zorder=3, edgecolor='white', linewidth=1.5)
        ax.text(x, y, name, fontsize=12, ha='center', va='center', color='white',
                fontweight='bold', zorder=4)

    # Элементы B
    for name, (x, y) in pos_B.items():
        color = '#e74c3c' if name in f_map.values() else '#bdc3c7'
        ax.scatter(x, y, s=600, color=color, zorder=3, edgecolor='white', linewidth=1.5)
        ax.text(x, y, name, fontsize=12, ha='center', va='center', color='white',
                fontweight='bold', zorder=4)

    # Стрелки
    for src, dst in f_map.items():
        x1, y1 = pos_A[src]
        x2, y2 = pos_B[dst]
        ax.add_patch(FancyArrowPatch((x1 + 0.15, y1), (x2 - 0.15, y2),
                                      arrowstyle='->', mutation_scale=15,
                                      color='#2c3e50', linewidth=1.5, zorder=2))

    ax.text(2.5, max(n_a, n_b) + 1.3, case['title'], fontsize=12, ha='center',
            fontweight='bold', color='#2c3e50')
    ax.text(2.5, -0.3, case['subtitle'], fontsize=10, ha='center', va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#fef9e7', edgecolor='#f39c12'))

    ax.set_xlim(-0.3, 5.3)
    ax.set_ylim(-1.2, max(n_a, n_b) + 2)
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle('Три типа отображений: инъекция, сюръекция, биекция', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Проверка кодом
def is_injective(f_map, A):
    return len(set(f_map.values())) == len(f_map)

def is_surjective(f_map, B):
    return set(f_map.values()) == set(B)

for case in cases:
    f = case['f']
    A = case['A']
    B = case['B']
    inj = is_injective(f, A)
    sur = is_surjective(f, B)
    bij = inj and sur
    print(f"{case['title']:35s}  inj={inj}  sur={sur}  bij={bij}")

---

## 7. Мощность множества — даже бесконечности бывают разные

### 7.1. Для конечных множеств

$|A|$ — просто число элементов. $|\{a, b, c\}| = 3$, $|\emptyset| = 0$.

### 7.2. Для бесконечных — сюрприз

Множество называется **счётным**, если его элементы можно **перенумеровать** натуральными числами. То есть существует биекция $f: \mathbb{N} \to A$.

| Множество | Счётное? | Мощность |
|---|---|---|
| $\mathbb{N}$ | ✅ да | $\aleph_0$ (алеф-ноль) |
| $\mathbb{Z}$ | ✅ да (перенумеруем: 0, 1, -1, 2, -2, ...) | $\aleph_0$ |
| $\mathbb{Q}$ | ✅ да (диагональное перечисление) | $\aleph_0$ |
| $\mathbb{R}$ | ❌ **нет** (доказательство Кантора) | $\mathfrak{c}$ (континуум) |
| $\mathbb{R}^n$ | ❌ нет | $\mathfrak{c}$ (та же!) |
| Мн-во всех функций $\mathbb{R} \to \mathbb{R}$ | ❌ нет | $> \mathfrak{c}$ (ещё больше!) |

### 7.3. Иерархия бесконечностей

$$|\mathbb{N}| = \aleph_0 < \mathfrak{c} = |\mathbb{R}| < |\mathcal{P}(\mathbb{R})| < |\mathcal{P}(\mathcal{P}(\mathbb{R}))| < \ldots$$

**Теорема Кантора:** $|A| < |\mathcal{P}(A)|$ для любого $A$. Бесконечностей бесконечно много, и нет «самой большой».

> **Зачем это в ML.** Пространство $\mathbb{R}^n$ при любом $n$ имеет **одну и ту же** мощность $\mathfrak{c}$. Это значит, что **любой вектор признаков можно «упаковать» в одно число** (теоретически). На практике это, конечно, невозможно из-за вычислительной точности, но концептуально — размерность не ограничивает «информационную ёмкость».

In [ ]:
# ============================================================
# Визуализация: счётное (Z) vs несчётное (R) множество
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Z — целые числа: «бусины на ниточке»
ax = axes[0]
ax.set_xlim(-5.5, 5.5)
ax.set_ylim(-0.5, 1.5)
ax.axhline(0.5, color='#3498db', linewidth=2, alpha=0.5)
for n in range(-5, 6):
    ax.scatter(n, 0.5, s=400, color='#3498db', zorder=3, edgecolor='white', linewidth=2)
    ax.text(n, 0.5, str(n), fontsize=10, ha='center', va='center', color='white',
            fontweight='bold', zorder=4)
# Подписи номеров
for i, n in enumerate(range(-5, 6)):
    ax.text(n, 0.05, f'#{i}', fontsize=8, ha='center', color='#7f8c8d')
ax.set_title('Z (целые) — СЧЁТНО: можно перенумеровать #0, #1, #2, ...', fontsize=12)
ax.axis('off')

# R — действительные: «плотная прямая»
ax = axes[1]
ax.set_xlim(-5.5, 5.5)
ax.set_ylim(-0.5, 1.5)
# Рисуем плотную прямую
gradient = np.linspace(-5, 5, 500).reshape(1, -1)
ax.imshow(gradient, extent=[-5, 5, 0.3, 0.7], aspect='auto', cmap='Reds', alpha=0.8)
ax.text(0, 0.5, 'ℝ — непрерывная прямая', fontsize=14, ha='center', va='center',
        color='white', fontweight='bold')
ax.text(0, 0.0, 'Между любыми двумя точками — бесконечно много других.\nНевозможно перенумеровать.',
        fontsize=10, ha='center', color='#7f8c8d')
ax.set_title('R (действительные) — НЕ СЧЁТНО: между любыми двумя — бесконечно много чисел', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.show()

print('Z счётно: 0, 1, -1, 2, -2, 3, -3, ... — есть алгоритм перечисления.')
print('R несчётно: доказывается диагональным аргументом Кантора (следующий раздел).')

---

## 8. Диагональный аргумент Кантора

Это **гениальное доказательство** того, что $\mathbb{R}$ не счётно. Профессор Кантор придумал его в 1891 году. Доказательство красивое — пять шагов.

### Постановка

Хотим доказать: множество $[0, 1]$ (все действительные числа от 0 до 1) **не счётно**.

### Доказательство от противного

**Шаг 1.** Предположим обратное: $[0, 1]$ счётно. Тогда его можно перенумеровать:

$$x_0, x_1, x_2, x_3, \ldots$$

Каждое $x_i$ — действительное число из $[0, 1]$.

**Шаг 2.** Запишем каждое $x_i$ как бесконечную десятичную дробь:

$$x_0 = 0.d_{0,0}d_{0,1}d_{0,2}d_{0,3}\ldots$$
$$x_1 = 0.d_{1,0}d_{1,1}d_{1,2}d_{1,3}\ldots$$
$$x_2 = 0.d_{2,0}d_{2,1}d_{2,2}d_{2,3}\ldots$$
$$\vdots$$

Здесь $d_{i,j}$ — $j$-я цифра $i$-го числа.

**Шаг 3.** Построим **новое** число $y = 0.e_0 e_1 e_2 \ldots$ по правилу:

$$e_i = \begin{cases} 1, & \text{если } d_{i,i} \neq 1 \\\\ 2, & \text{если } d_{i,i} = 1 \end{cases}$$

То есть смотрим на **диагональ** $d_{0,0}, d_{1,1}, d_{2,2}, \ldots$ и для каждой цифры берём **другую**.

**Шаг 4.** Тогда $y \in [0, 1]$, но $y \neq x_i$ ни при каком $i$, потому что на $i$-й позиции у них разные цифры: $e_i \neq d_{i,i}$.

**Шаг 5.** Противоречие! Мы предположили, что перенумеровали все числа из $[0, 1]$, но построили $y \in [0, 1]$, которого в списке нет. Значит, предположение неверно, и $[0, 1]$ **не счётно**. $\blacksquare$

### Почему цифры 1 и 2?

Чтобы избежать проблемы двойного представления: $0.4999\ldots = 0.5$. Если бы мы выбирали случайные цифры, могли бы случайно «наткнуться» на такое равенство. Используя только 1 и 2, мы гарантируем однозначность.

> **Зачем это в ML.** Это доказательство — пример **конструктивного противоречия**. Похожий приём используется в теореме о невозможности: нельзя построить универсальный алгоритм, который решает «остановится ли программа». В ML — нельзя построить идеальный мета-обучатель (no free lunch theorem).

In [ ]:
# ============================================================
# Визуализация: диагональный аргумент Кантора
# ============================================================
# Допустим, мы «перенумеровали» 8 чисел из [0,1]:
fake_listing = [
    '1234567890',
    '9876543210',
    '1111111111',
    '2222222222',
    '3141592653',
    '2718281828',
    '0000000000',
    '5555555555',
]

fig, ax = plt.subplots(figsize=(11, 6))

n = len(fake_listing)
# Рисуем таблицу чисел, подсвечивая диагональ
for i, s in enumerate(fake_listing):
    for j, digit in enumerate(s):
        if i == j:
            # Диагональ — красным
            color = '#e74c3c'
            fontsize = 16
            fontweight = 'bold'
        else:
            color = '#2c3e50'
            fontsize = 13
            fontweight = 'normal'
        ax.text(j * 0.8, -i, digit, fontsize=fontsize, ha='center', va='center',
                color=color, fontweight=fontweight, family='monospace')
    # Подпись слева
    ax.text(-1.5, -i, f'x_{i} = 0.', fontsize=11, ha='right', va='center',
            fontweight='bold', color='#2c3e50')

# Строим диагональное число y
y_digits = []
for i in range(n):
    d = fake_listing[i][i]
    new_d = '1' if d != '1' else '2'  # 1 если d != 1, иначе 2
    y_digits.append(new_d)

y_str = ''.join(y_digits)
ax.text(n * 0.8 + 1.5, -n/2 + 0.5, 'y = 0.' + y_str, fontsize=16, ha='left', va='center',
        fontweight='bold', color='#27ae60', family='monospace',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#eafaf1', edgecolor='#27ae60'))

ax.text(n * 0.8 + 1.5, -n/2 - 0.5, '← число, которого\nнет в списке!', fontsize=11, ha='left',
        color='#27ae60', fontweight='bold')

# Подпись
ax.text(-1, -n - 1, 'Диагональ (красным) — те цифры, которые мы «портим» при построении y.\n'
                    'y отличается от каждого x_i в i-й позиции → y не входит в список.',
        fontsize=11, color='#7f8c8d', style='italic')

# Линия-диагональ
for i in range(n):
    ax.plot([i * 0.8, i * 0.8], [-i - 0.3, -i + 0.3], color='#e74c3c', alpha=0.3, linewidth=2)

ax.set_xlim(-3, n * 0.8 + 7)
ax.set_ylim(-n - 2, 1.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Диагональный аргумент Кантора: строим число y, которого нет в списке',
             fontsize=13, pad=15)
plt.tight_layout()
plt.show()

# Проверка: y действительно отличается от каждого x_i в i-й позиции
print('Проверка различия y и x_i в i-й позиции:')
for i, s in enumerate(fake_listing):
    print(f'  y[{i}]={y_str[i]}  vs  x_{i}[{i}]={s[i]}  → различаются? {y_str[i] != s[i]}')

---

## 9. Задачи из лекции ВК — разбираем все шесть

### Задача 1. Мощность $\mathbb{R}^k$

**Утверждение:** $|\mathbb{R}^k| = \mathfrak{c}$ (континуум) для любого $k \geq 1$.

**Идея:** действительное число = бесконечная десятичная дробь $0.x_1 x_2 x_3 \ldots$ Набор из $k$ чисел — это $k$ таких дробей. **Чередуем их цифры**:

$$X = \begin{pmatrix} 0.x^{(1)}_1 x^{(1)}_2 \ldots \\\\ \vdots \\\\ 0.x^{(k)}_1 x^{(k)}_2 \ldots \end{pmatrix} \quad \mapsto \quad Y = 0.x^{(1)}_1 x^{(2)}_1 \ldots x^{(k)}_1 x^{(1)}_2 x^{(2)}_2 \ldots x^{(k)}_2 \ldots$$

Это **инъекция** $\mathbb{R}^k \to \mathbb{R}$. Обратная инъекция тривиальна: $\eta: x \mapsto (x, 0, \ldots, 0)$. По **теореме Кантора-Бернштейна** (есть инъекции в обе стороны → есть биекция) получаем $|\mathbb{R}^k| = |\mathbb{R}| = \mathfrak{c}$.

### Задача 2. Мощность $\mathbb{Q}^k$

**Утверждение:** $|\mathbb{Q}^k| = \aleph_0$ (счётно).

**Идея:** $\mathbb{Q}$ счётно (диагональное перечисление: $1/1, 2/1, 1/2, 3/1, 1/3, 2/3, 4/1, 1/4, 3/4, \ldots$ с пропуском сократимых). Для $\mathbb{Q}^k$ то же самое делается покоординатно. **Счётное объединение счётных = счётное.**

### Задача 3. Конечные последовательности натуральных чисел

**Утверждение:** множество $\bigcup_{k=1}^\infty \mathbb{N}^k$ **счётно**.

**Идея:** фиксируем длину $k$ и сумму $S$. Конечное число последовательностей длины $k$ с суммой $S$. Перебираем по $S$, потом по $k$. Пример для $k=3$:

$$(0,0,0), (0,0,1), (0,1,0), (1,0,0), (0,0,2), (0,2,0), (2,0,0), (0,1,1), (1,0,1), (1,1,0), \ldots$$

### Задача 4. Убывающие последовательности натуральных

**Утверждение:** счётно.

**Идея:** натуральные не убывают до бесконечности, поэтому любая убывающая последовательность **конечна** (заканчивается в 0). Это подмножество множества из задачи 3.

### Задача 5. Невозрастающие последовательности натуральных

**Утверждение:** счётно.

**Идея:** последовательность может быть бесконечной, но начинается с $k$ и в какой-то момент **стабилизируется** на константе $m \leq k$. Фиксируем старт $k$ и момент стабилизации $t$ — таких последовательностей конечное число. Объединение по $t$ счётно, по $k$ — счётное объединение счётных.

### Задача 6. Все последовательности натуральных $\mathbb{N}^\mathbb{N}$

**Утверждение:** **континуум** $\mathfrak{c}$ (не счётно!).

**Идея:** диагональный аргумент, как для $\mathbb{R}$. Пусть множество счётно: $a_0, a_1, \ldots$ где $a_i: \mathbb{N} \to \mathbb{N}$. Строим $f$, отличающуюся от $a_i$ в $i$-й точке:

$$f(n) = a_n(n) + f(n-1) + 1$$

(с $f(0) = a_0(0) + 1$). Тогда $f$ возрастает и не совпадает ни с одним $a_i$. Противоречие. $\blacksquare$

In [ ]:
# ============================================================
# Демонстрация: счётность Q через диагональное перечисление
# ============================================================
from fractions import Fraction

def enumerate_Q(max_total=15):
    """Перечисляет неотрицательные рациональные p/s по росту суммы p+s."""
    seen = set()
    result = []
    for total in range(1, max_total + 1):
        for p in range(0, total + 1):
            s = total - p
            if s == 0:
                continue
            f = Fraction(p, s)  # автоматически сокращает
            if f not in seen:
                seen.add(f)
                result.append(f)
    return result

q_list = enumerate_Q(15)
print(f'Первые 25 рациональных чисел в диагональном перечислении:')
print(f'  {q_list[:25]}')
print(f'Всего дробей с p+s ≤ 15: {len(q_list)}')
print('Каждое рациональное рано или поздно появится → Q счётно.')

# Визуализация
fig, ax = plt.subplots(figsize=(9, 7))
for total in range(1, 11):
    points = []
    for p in range(0, total + 1):
        s = total - p
        if s == 0:
            continue
        points.append((p, s))
    if points:
        xs = [p for p, s in points]
        ys = [s for p, s in points]
        ax.scatter(xs, ys, s=100, zorder=3, color=f'C{total}')
        ps = sorted(points)
        ax.plot([p for p, s in ps], [s for p, s in ps], 'gray', alpha=0.4)
        # Подпись суммы
        ax.text(max(xs) + 0.3, max(ys), f'p+s={total}', fontsize=8, color=f'C{total}')

ax.set_xlabel('p (числитель)')
ax.set_ylabel('s (знаменатель)')
ax.set_title('Диагональное перечисление Q: p/s по росту p+s')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 10. Зачем всё это в ML и нейросетях?

Профессор сводит концы с концами.

### 10.1. Множества — это датасеты

Любой датасет — это множество объектов. MNIST — множество картинок 28×28. ImageNet — множество картинок с подписями. Текстовый корпус — множество токенов.

Когда ты говоришь «у меня 1 миллион примеров» — ты говоришь $|D| = 10^6$. Когда ты делаешь `train_test_split` — ты разбиваешь множество на две части: $D = D_{train} \sqcup D_{test}$ (дизъюнктное объединение).

### 10.2. Функции — это нейросети

Нейросеть с весами $\theta$ — это отображение:

$$f_\theta: \mathbb{R}^n \to \mathbb{R}^m$$

где $n$ — размер входа, $m$ — размер выхода. Например, ResNet-50: $f: \mathbb{R}^{224 \times 224 \times 3} \to \mathbb{R}^{1000}$ (картинка → вероятности классов).

### 10.3. Loss-функция — это функционал

Loss $L$ берёт функцию (нейросеть) и возвращает число — «насколько плоха эта сеть»:

$$L: \mathcal{F} \to \mathbb{R}, \quad L(f) = \frac{1}{|D|} \sum_{(x, y) \in D} \ell(f(x), y)$$

Это **функционал** — отображение из пространства функций в числа.

### 10.4. Градиент — это оператор

Градиент $\nabla$ берёт функцию и возвращает другую функцию (градиентную):

$$\nabla: L \mapsto \nabla L$$

В отличие от обычной производной (число), градиент — это вектор частных производных, то есть функция, возвращающая вектор. **Оператор**.

### 10.5. Эмбеддинги и инъекции

Хочешь, чтобы разным токенам соответствовали **разные** векторы? Это **инъекция** $\text{токен} \to \mathbb{R}^d$. Если токенов $|V| \leq 2^d$, можно построить инъекцию. Если $|V| > 2^d$ — принцип Дирихле, коллизий не избежать.

### 10.6. Автокодеры и биекции

Автокодер учит биекцию «оригинал ↔ сжатое представление». Если сжатое представление меньше оригинала — это **не** биекция (не инъекция). Поэтому автокодер всегда теряет информацию, и loss не равен нулю.

### 10.7. Мощность и размерность

$|\mathbb{R}^n| = \mathfrak{c}$ для любого $n$. Это значит, что **размерность не ограничивает информационную ёмкость**. Теоретически, любой вектор признаков можно «упаковать» в одно число. Практически — из-за конечной точности float это не работает, но концептуально важно: нейросеть может выучить отображение из $\mathbb{R}^{1000}$ в $\mathbb{R}^{10}$ без потери информации (если данные «хорошие»).

### 10.8. No Free Lunch Theorem

Теорема «о бесплатном сыре» — аналог диагонального аргумента Кантора в ML. Не существует универсального алгоритма, который лучше всех на любых данных. Доказательство — от противного, конструктивное, в духе Кантора.

---

> **Резюме профессора.** Если ты понял множества и функции — ты понял **язык**, на котором написана вся математика ML. Сами формулы нейросетей — это просто конкретные функции в этом языке.

---

## 11. Проверочный тест

**Правила:** 10 вопросов, по 1 баллу за каждый правильный ответ. Результаты считаются автоматически.

| Баллов | Оценка |
|---|---|
| 9–10 | Отлично! Можешь идти дальше |
| 7–8 | Хорошо, но есть пробелы |
| 5–6 | Удовлетворительно — перечитай темы |
| 0–4 | Срочно перечитай лекцию с начала |

Запусти следующую ячейку и отвечай на вопросы. После последнего вопроса увидишь итоговый счёт.

In [ ]:
# ============================================================
# Интерактивный тест на проверку усвоения материала
# ============================================================
# Запусти эту ячейку (Shift+Enter), чтобы пройти тест.

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Вопросы: (вопрос, [варианты], индекс правильного ответа, пояснение)
QUESTIONS = [
    {
        'q': 'Что означает запись $x \\in A$?',
        'options': [
            '$x$ не лежит в $A$',
            '$x$ лежит в $A$ (принадлежит множеству $A$)',
            '$x$ равен $A$',
            '$A$ — подмножество $x$',
        ],
        'correct': 1,
        'expl': 'Символ $\\in$ читается «принадлежит». $x \\in A$ — «$x$ лежит в $A$».'
    },
    {
        'q': 'Чему равно $|\\{1, 1, 2, 3\\}|$?',
        'options': ['1', '2', '3', '4'],
        'correct': 2,
        'expl': 'В множестве нет дубликатов: $\\{1, 1, 2, 3\\} = \\{1, 2, 3\\}$. Размер = 3.'
    },
    {
        'q': 'Какое из множеств счётное?',
        'options': [
            '$\\mathbb{R}$ (действительные числа)',
            '$\\mathbb{N}$ (натуральные числа)',
            'Интервал $[0, 1]$',
            'Множество всех последовательностей натуральных чисел',
        ],
        'correct': 1,
        'expl': 'Только $\\mathbb{N}$ счётно — по определению (его элементы можно перенумеровать). $\\mathbb{R}$, $[0,1]$ и $\\mathbb{N}^\\mathbb{N}$ несчётны (доказано Кантором).'
    },
    {
        'q': 'Что такое инъекция?',
        'options': [
            'Отображение, где все $y$ достижимы',
            'Отображение, где разным $x$ соответствуют разные $y$',
            'Взаимно однозначное соответствие',
            'Отображение, у которого есть обратное',
        ],
        'correct': 1,
        'expl': 'Инъекция: $f(x_1) = f(x_2) \\Rightarrow x_1 = x_2$. Разные $x$ → разные $y$.'
    },
    {
        'q': 'Какой закон де Моргана верный?',
        'options': [
            '$\\overline{A \\cup B} = \\overline{A} \\cup \\overline{B}$',
            '$\\overline{A \\cap B} = \\overline{A} \\cap \\overline{B}$',
            '$\\overline{A \\cup B} = \\overline{A} \\cap \\overline{B}$',
            '$A \\cup B = A \\cap B$',
        ],
        'correct': 2,
        'expl': 'Дополнение объединения = пересечение дополнений: $\\overline{A \\cup B} = \\overline{A} \\cap \\overline{B}$.'
    },
    {
        'q': 'Сколько подмножеств у множества из 5 элементов?',
        'options': ['5', '10', '25', '32'],
        'correct': 3,
        'expl': 'По теореме: $|\\mathcal{P}(A)| = 2^{|A|}$. Если $|A| = 5$, то $2^5 = 32$.'
    },
    {
        'q': 'Чему равно $|\\mathbb{R}^3|$?',
        'options': [
            '$\\aleph_0$ (счётно)',
            '$\\mathfrak{c}$ (континуум)',
            '$3 \\mathfrak{c}$',
            '$2^{\\mathfrak{c}}$',
        ],
        'correct': 1,
        'expl': 'По задаче 1 лекции: $|\\mathbb{R}^k| = \\mathfrak{c}$ для любого $k$. Это доказывается чередованием цифр + Кантор-Бернштейн.'
    },
    {
        'q': 'В чём суть диагонального аргумента Кантора?',
        'options': [
            'Строим число, которое отличается от каждого $x_i$ в $i$-й позиции',
            'Сравниваем мощности множеств через инъекции',
            'Считаем элементы по диагонали матрицы',
            'Доказываем счётность $\\mathbb{Q}$',
        ],
        'correct': 0,
        'expl': 'Строим $y$, у которого $i$-я цифра отличается от $i$-й цифры $x_i$. Тогда $y$ не входит в список.'
    },
    {
        'q': 'Что такое булеан $\\mathcal{P}(A)$?',
        'options': [
            'Множество всех элементов $A$',
            'Множество всех подмножеств $A$',
            'Объединение $A$ с самим собой',
            'Дополнение $A$',
        ],
        'correct': 1,
        'expl': 'Булеан — множество всех подмножеств $A$ (включая $\\emptyset$ и само $A$). $|\\mathcal{P}(A)| = 2^{|A|}$.'
    },
    {
        'q': 'Какая связь между нейросетью и понятием функции?',
        'options': [
            'Нейросеть — это не функция, а алгоритм',
            'Нейросеть — это отображение $\\mathbb{R}^n \\to \\mathbb{R}^m$',
            'Нейросеть — это функционал',
            'Нейросеть — это оператор',
        ],
        'correct': 1,
        'expl': 'Нейросеть с весами $\\theta$ — это отображение $f_\\theta: \\mathbb{R}^n \\to \\mathbb{R}^m$. Loss — функционал, градиент — оператор.'
    },
]

# --- Состояние теста ---
state = {'index': 0, 'score': 0, 'answers': []}

# --- Виджеты ---
question_html = widgets.HTML()
options_radio = widgets.RadioButtons(options=[], layout=widgets.Layout(width='80%'))
feedback_html = widgets.HTML()
next_button = widgets.Button(description='Ответить →', button_style='primary',
                              layout=widgets.Layout(width='200px'))
progress_label = widgets.HTML()

def render_question():
    """Отрисовывает текущий вопрос."""
    i = state['index']
    q = QUESTIONS[i]
    question_html.value = f'<h3 style="color:#2c3e50">Вопрос {i+1} из {len(QUESTIONS)}</h3><p style="font-size:14px">{q["q"]}</p>'
    options_radio.options = q['options']
    options_radio.value = None  # сброс
    feedback_html.value = ''
    next_button.description = 'Ответить →'
    progress_label.value = f'<small>Счёт: {state["score"]}/{state["index"]}</small>'

def on_next(b):
    """Обработка нажатия на кнопку."""
    i = state['index']
    q = QUESTIONS[i]

    if options_radio.value is None:
        feedback_html.value = '<span style="color:#e74c3c">⚠️ Выбери вариант ответа!</span>'
        return

    chosen = q['options'].index(options_radio.value)
    correct = chosen == q['correct']

    if correct:
        state['score'] += 1
        feedback_html.value = f'<div style="background:#eafaf1;padding:10px;border-left:4px solid #27ae60"><b style="color:#27ae60">✓ Верно!</b><br><small>{q["expl"]}</small></div>'
    else:
        correct_opt = q['options'][q['correct']]
        feedback_html.value = f'<div style="background:#fdedee;padding:10px;border-left:4px solid #e74c3c"><b style="color:#e74c3c">✗ Неверно.</b> Правильный ответ: <b>{correct_opt}</b><br><small>{q["expl"]}</small></div>'

    state['answers'].append({'q': q['q'], 'chosen': chosen, 'correct': q['correct'], 'ok': correct})

    if i + 1 < len(QUESTIONS):
        next_button.description = 'Следующий вопрос →'
        # Меняем обработчик: при следующем нажатии переходим к новому вопросу
        next_button.on_click(go_next, remove=True)
        next_button.on_click(go_next)
    else:
        next_button.description = 'Показать результат 🏁'
        next_button.on_click(show_results, remove=True)
        next_button.on_click(show_results)

def go_next(b):
    state['index'] += 1
    next_button.on_click(go_next, remove=True)
    next_button.on_click(on_next)
    render_question()

def show_results(b):
    """Финальный экран с результатом."""
    score = state['score']
    total = len(QUESTIONS)
    pct = score / total * 100

    if pct >= 90:
        grade, color, emoji = 'Отлично!', '#27ae60', '🎉'
    elif pct >= 70:
        grade, color, emoji = 'Хорошо', '#3498db', '👍'
    elif pct >= 50:
        grade, color, emoji = 'Удовлетворительно', '#f39c12', '🤔'
    else:
        grade, color, emoji = 'Нужно перечитать', '#e74c3c', '📚'

    html = f'''
    <div style="background:{color};color:white;padding:20px;border-radius:10px;text-align:center">
        <h2>{emoji} {grade}</h2>
        <p style="font-size:18px">Твой результат: <b>{score} / {total}</b> ({pct:.0f}%)</p>
    </div>
    <h4 style="margin-top:20px">Разбор ответов:</h4>
    <ol>
    '''
    for ans in state['answers']:
        icon = '✓' if ans['ok'] else '✗'
        clr = '#27ae60' if ans['ok'] else '#e74c3c'
        html += f'<li style="color:{clr}">{icon} {ans["q"][:80]}...</li>'
    html += '</ol>'

    clear_output()
    display(widgets.HTML(html))
    display(widgets.Button(description='Пройти заново', button_style='info',
                            layout=widgets.Layout(width='200px')))

# Старт
next_button.on_click(on_next)
render_question()

display(widgets.VBox([
    progress_label,
    question_html,
    options_radio,
    feedback_html,
    next_button,
]))

---

## Финал лекции

> Вот и всё, студент. Ты прошёл:
>
> - множества и операции над ними
> - подмножества и булеан
> - отображения и их типы (инъекция, сюръекция, биекция)
> - мощность и счётность
> - диагональный аргумент Кантора
> - шесть задач из лекции ВК
> - связь всего этого с нейросетями
> - тест на 10 вопросов
>
> Если прошёл тест на 7+ — поздравляю, ты освоил фундамент. Дальше пойдёт **линейная алгебра** (векторы, матрицы, линейные отображения — это специальные функции!). И там уже будет понятно, зачем нужны все эти множества.
>
> Если не прошёл — ничего страшного. Георг Кантор сам долго не мог доказать свои теоремы. Перечитай лекцию, посмотри на картинки, пройди тест ещё раз. Математика — это как спортзал: повторения важнее таланта.
>
> Удачи! *Профессор.*

---

## Полезные ссылки

- **Курс ВК «Математика для ML»:** https://education.vk.company/curriculum/program/lesson/31702/
- **AllCups (задания курса):** https://allcups.run/
- **Halmos, «Naive Set Theory»** — классическое доступное введение
- **Кантор, «К теории многообразий»** (1878) — оригинальная статья о мощностях

> *Ноутбук создан по материалам Лекции 1 курса «Математика для машинного обучения и анализа данных» (ВКонтакте Образование, 2025).*
> *Соглашение курса: $0 \\in \\mathbb{N}$.*